## EDA: Condutividade — `condut_tejo_loc_zvt.csv`

Caminho do dataset:
`/Users/diogopinto/Documents/Usar/git_clep/clepsydra_isa/EDA /data/condut_tejo_loc_zvt.csv`

Objetivo: caracterizar séries por ponto (cobertura temporal, estatísticas, outliers), localização, frequência e lacunas; exportar resumos — tudo dentro de `EDA`.


### 1) Setup de bibliotecas e configuração de paths


In [ ]:
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except Exception:
    sns = None

DATA_FILE = Path("/Users/diogopinto/Documents/Usar/git_clep/clepsydra_isa/EDA /data/condut_tejo_loc_zvt.csv")
assert DATA_FILE.exists(), f"CSV not found: {DATA_FILE}"

pd.options.display.float_format = "{:.3f}".format
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print(f"Using file: {DATA_FILE}")


### 2) Leitura do CSV e preparação de colunas

#### 2.1 Explicação do código
- `parse_dates` para `data`.
- Renomeação: `condcamp20c` -> `ec_20c_uScm` (condutividade a 20ºC), `condutividade` -> `ec_field_uScm` (se existir).
- `site_id`/`site_label` (`codigo | y_x`).


In [ ]:
parse_dates = ["data"]
df = pd.read_csv(
    DATA_FILE,
    parse_dates=parse_dates,
    dtype={
        "codigo": "string",
        "condutividade": "float32",
        "condcamp20c": "float32",
        "coord_x_m": "float32",
        "coord_y_m": "float32",
        "altitude_m": "float32",
        "sistema_aquifero": "string",
        "estado": "string",
        "freguesia": "string",
    },
)

rename_map = {
    "data": "date",
    "codigo": "well_code",
    "condutividade": "ec_field_uScm",
    "condcamp20c": "ec_20c_uScm",
    "coord_x_m": "x_m",
    "coord_y_m": "y_m",
}
df = df.rename(columns=rename_map).sort_values("date").reset_index(drop=True)

# Derivadas
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["site_id"] = df["y_m"].round(2).astype(str) + "_" + df["x_m"].round(2).astype(str)
df["site_label"] = df["well_code"].astype(str) + " | " + df["site_id"]

print(df.head()); df.info()


### 3) Cobertura temporal, frequência e lacunas por ponto

- Estatísticas de `ec_20c_uScm`/`ec_field_uScm`.
- Inferência de frequência e períodos em falta por `site_id`.
- Duplicados por `date`×`site_id`.


In [ ]:
value_cols = [c for c in ["ec_20c_uScm", "ec_field_uScm"] if c in df.columns]
print(df[value_cols].describe(percentiles=[0.01, 0.05, 0.95, 0.99]))

dups = df[df.duplicated(["date","site_id"], keep=False)]
print(f"Duplicate timestamps (per site): {dups.shape[0]}")

# Frequência e lacunas (diário/mensal/irregular) — funções simples

def infer_frequency(ts: pd.Series) -> str:
    if ts.size < 2:
        return "unknown"
    diffs = ts.sort_values().diff().dropna()
    if diffs.empty:
        return "unknown"
    median = diffs.median(); days = median / pd.Timedelta(days=1)
    if 27 <= days <= 31:
        return "monthly"
    if 0.8 <= days <= 1.2:
        return "daily"
    return "irregular"


def compute_missing(df_site: pd.DataFrame, freq_label: str) -> tuple[int, list]:
    if df_site.empty:
        return 0, []
    start = df_site["date"].min().normalize(); end = df_site["date"].max().normalize()
    freq = "MS" if freq_label=="monthly" else ("D" if freq_label=="daily" else None)
    if freq is None:
        return 0, []
    full_range = pd.date_range(start=start, end=end, freq=freq)
    missing = full_range.difference(df_site["date"].dt.normalize().unique())
    return int(missing.size), [str(x.date()) for x in list(missing[:5])]

rows = []
for sid, d in df.groupby("site_id"):
    f = infer_frequency(d["date"])
    mcount, mfirst = compute_missing(d, f)
    rows.append({
        "site_id": sid,
        "site_label": d["site_label"].iloc[0],
        "freq_label": f,
        "missing_count": mcount,
        "missing_first_5": ", ".join(mfirst),
    })
site_frequency_summary = pd.DataFrame(rows)
display(site_frequency_summary.head())


### 4) Outliers e gráficos

- Deteção IQR + Z-score para a variável de condutividade disponível.
- Gráfico agregado e dropdown por poço.


In [ ]:
target_col = "ec_20c_uScm" if "ec_20c_uScm" in df.columns else "ec_field_uScm"

def detect_outliers_iqr(series: pd.Series, factor: float = 1.5) -> pd.Series:
    q1 = np.nanpercentile(series, 25); q3 = np.nanpercentile(series, 75)
    iqr = q3 - q1; lower = q1 - factor * iqr; upper = q3 + factor * iqr
    return (series < lower) | (series > upper)


def detect_outliers_zscore(series: pd.Series, threshold: float = 3.0) -> pd.Series:
    mu = np.nanmean(series); sigma = np.nanstd(series)
    if sigma == 0 or np.isnan(sigma):
        return pd.Series(False, index=series.index)
    z = (series - mu) / sigma; return z.abs() > threshold

flags = detect_outliers_iqr(df[target_col]) | detect_outliers_zscore(df[target_col])
df["is_outlier_ec"] = flags
print(f"{target_col} outliers: {flags.sum()}")

plt.figure(figsize=(14,5))
plt.plot(df["date"], df[target_col], alpha=0.6, label=f"ALL SITES · {target_col}")
plt.scatter(df.loc[flags, "date"], df.loc[flags, target_col], s=10, color="black", label="outlier")
plt.legend(); plt.xlabel("date"); plt.ylabel(f"{target_col} [µS/cm]")
plt.title("Condutividade (todas as observações)"); plt.tight_layout(); plt.show()

import ipywidgets as widgets
from IPython.display import display, clear_output
sites = sorted(df["site_id"].unique()); labels = df.groupby("site_id")["site_label"].first().to_dict()
opts = [(labels[s], s) for s in sites]; out = widgets.Output(); dd = widgets.Dropdown(options=opts, value=sites[0], description="site:")

def plot_site(sid: str):
    d = df[df["site_id"] == sid].sort_values("date")
    lab = labels.get(sid, sid)
    plt.figure(figsize=(14,5))
    plt.plot(d["date"], d[target_col], label=lab)
    if "is_outlier_ec" in d.columns:
        m = d["is_outlier_ec"]; plt.scatter(d.loc[m, "date"], d.loc[m, target_col], s=10, color="black", label="outlier")
    plt.legend(); plt.xlabel("date"); plt.ylabel(f"{target_col} [µS/cm]")
    plt.title(f"Condutividade ({lab})"); plt.tight_layout(); plt.show()

def on_change(c):
    if c["name"] == "value":
        with out:
            clear_output(wait=True); plot_site(c["new"]) 

dd.observe(on_change, names="value"); display(dd)
with out: plot_site(dd.value)
display(out)


### 5) Resumo por ponto e export

- Localização e métricas básicas.
- Junta frequência e lacunas por poço.
- Export para `EDA /scripts/condutividade/resources/`.


In [ ]:
agg = {target_col: ["mean", "median", "min", "max"]}
location_summary = (
    df.groupby(["site_id", "site_label", "y_m", "x_m"]).agg(
        n_obs=("date", "count"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        mean_ec=(target_col, "mean"),
        p95_ec=(target_col, lambda s: np.nanpercentile(s, 95)),
    ).reset_index()
)

location_summary = location_summary.merge(site_frequency_summary, on=["site_id", "site_label"], how="left")

display(location_summary.head())

from pathlib import Path
out_dir = Path("/Users/diogopinto/Documents/Usar/git_clep/clepsydra_isa/EDA /scripts/condutividade/resources")
out_dir.mkdir(parents=True, exist_ok=True)

summary_stats = df[[target_col]].describe().T
summary_stats.loc["meta_unique_points", "count"] = location_summary.shape[0]

location_summary.to_csv(out_dir / "condut_location_summary.csv", index=False)
site_frequency_summary.to_csv(out_dir / "condut_frequency_gaps_by_site.csv", index=False)
summary_stats.to_csv(out_dir / "condut_summary_stats.csv")

print(f"Saved summaries in: {out_dir}")
